# 04a — Lakeflow Connect: Managed SQL Server Connector

Trainer-driven demo. Participants follow along and run the verification cells.

RetailHub's finance team keeps its wholesale channel in an operational Azure SQL Database
(the AdventureWorksLT sample, with CDC enabled). We do not hand-write JDBC extract jobs for it.
Lakeflow Connect, the managed SQL Server connector, ingests it straight into Unity Catalog bronze tables.

| Section | What happens | Who acts |
|---|---|---|
| A | Decision framework: which ingestion tool when | discussion |
| B | Prerequisites and architecture (connection → gateway → ingestion pipeline) | theory |
| C | UI walkthrough: create the connection and the ingestion pipeline | TRAINER |
| D | Verify the landed tables with SQL | **everyone** |
| E | Live CDC round-trip: UPDATE in the source, the change appears in bronze | TRAINER + everyone |
| F | Exam call-out and cost note | wrap-up |

> **Trainer:** the source database, connection values, dry run, grants and teardown for this demo are in `docs/INSTRUCTOR_GUIDE.md` §0, item 5.

## A. Decision framework: picking the right ingestion tool

Ingestion is 21% of the exam. A typical question reads: *"Data lives in X, lands in Y, with Z constraints. Which tool?"*
Start from the source type, then check the effort and the incremental support.

| Tool | Source type | Effort | Incremental? | When to pick it |
|---|---|---|---|---|
| Auto Loader (`cloudFiles`), a Lakeflow Connect *standard* connector | Files in object storage (S3/ADLS/GCS/Volumes): JSON, CSV, Parquet, Avro… | Low (code) | Yes: file discovery and checkpoints | Continuous or scheduled file feeds; schema evolution; exactly-once file ingestion |
| Lakeflow Connect standard connectors | Cloud object storage (Auto Loader, row above), SFTP and message buses (Kafka, Kinesis, Pub/Sub, Pulsar). You write the reader in code with Spark Declarative Pipelines or Structured Streaming | Low-medium (code) | Yes: checkpoints or offsets | File feeds, SFTP drops and streaming sources where you control the reader in pipeline code |
| Lakeflow Connect managed connectors | SaaS apps and databases: Salesforce, Workday, ServiceNow, Google Analytics, **SQL Server / Azure SQL** … | Low (UI/API, no code) | Yes: snapshots and **CDC / change tracking** | Fully managed ingestion of enterprise apps and databases into UC, with minimal engineering |
| Partner tools (Fivetran, Qlik, Informatica…) | Hundreds of long-tail sources | Low (SaaS) | Yes (depends on the tool) | Lakeflow Connect does not cover the source; you already have a partner licence or team |
| JDBC in a notebook (`spark.read.jdbc`) | Any JDBC database | Medium-high (you own all of it) | No: full reads unless you build watermarking yourself | One-off exploration, tiny lookup tables. Avoid it for production feeds |

Rules of thumb:
- Files land in storage → Auto Loader.
- An enterprise application or an operational database with a managed connector → Lakeflow Connect managed connector.
- No connector exists → a partner tool before hand-rolled JDBC.
- Hand-written JDBC is the last resort: no CDC, no checkpoints, and you own retries and schema drift.

<img src="../../../assets/images/nb02-ingestion-decision.png" alt="Decision tree from the source type to the ingestion tool: files to COPY INTO or Auto Loader, SaaS and databases to Lakeflow Connect" width="800">

### Which would you pick? (3 mini-scenarios)

Pick a tool for each scenario before you read the answer.

Scenario 1. RetailHub's POS vendor drops ~200 gzipped JSON files per hour into an ADLS container.
You need them in bronze within minutes, and schema evolution must be handled.

> **Answer:** Auto Loader, in streaming or triggered batch mode. It is built for files in object storage. File notifications or directory listing, together with checkpoints, give exactly-once ingestion.

Scenario 2. The sales team works in Salesforce. Leadership wants opportunities and accounts refreshed in the lakehouse every hour, and there is no engineering budget for custom code.

> **Answer:** The Lakeflow Connect managed Salesforce connector. It is fully managed and incremental, and you configure it in the UI. JDBC is not even possible here. A partner tool would work, but it is the second choice when a first-party managed connector exists.

Scenario 3. A legacy on-prem Informix database must be replicated daily. Lakeflow Connect has no connector for it, and the company already licenses Fivetran.

> **Answer:** A partner tool (Fivetran). No managed connector exists. With hand-written JDBC you would build the incremental logic and the monitoring yourself.

## B. Prerequisites and architecture

The managed SQL Server connector is built from three objects that work together (items 2–4 below):

<img src="../../../assets/images/m03a-architecture.png" width="800">

1. The source is SQL Server (here Azure SQL DB `AdventureWorksLT`) with change tracking or CDC on. The connector logs in as `lakeflow_connect`, a low-privilege user (`SELECT` + metadata/CDC grants).
2. A UC connection (type SQL Server) holds the host and the credentials, governed in Unity Catalog.
3. An ingestion gateway pipeline on classic compute extracts the snapshot and the changes from the source. It stages them in a UC volume.
4. An ingestion pipeline on serverless applies the staged changes to the target streaming tables in `<catalog>.lakeflow_connect`.

Prerequisites checklist:
- Unity Catalog and serverless compute are enabled in the workspace. You need permission to create classic clusters, because the gateway runs on classic compute.
- Unity Catalog privileges: `CREATE CONNECTION` on the metastore, and `USE CATALOG`/`CREATE SCHEMA`/`CREATE TABLE` + `CREATE VOLUME` on the staging and target locations.
- The source database has CDC (`sys.sp_cdc_enable_db` / `..._table`) or change tracking enabled. In this training CDC is on for the four `SalesLT` tables. Databricks recommends change tracking for tables with a primary key. If both are enabled, the connector uses change tracking.
- The gateway has a network path to the SQL endpoint. Here the Azure SQL firewall allows Azure services.

The gateway pipeline runs on dedicated compute *inside your network path*. It does the heavy work on the source side:
the initial snapshot and reading the change feed. The serverless ingestion pipeline is decoupled from it and only merges
the staged changes into Unity Catalog tables. One gateway can serve the whole class, because participants only *read* the landed tables.

## C. TRAINER: UI walkthrough

> Participants: watch which objects the trainer creates. You will query the results in section D.

### C.1 Create the UC connection

1. Go to **Catalog** → **External Data** → **Connections** → **Create connection**.
2. Set **Connection name** to `retailhub_sqlserver` and **Connection type** to `SQL Server`.
3. For authentication, use user/password. Enter host `<server>.database.windows.net`, port `1433`,
   user `lakeflow_connect` (a low-privilege login: `SELECT` + metadata/CDC grants) and its password.
4. Click **Test connection**, then **Create**. The connection is now a governed UC object (`SHOW CONNECTIONS`).

### C.2 Create the ingestion pipeline (wizard)
1. Open **Data Ingestion** (**+ New → Add data**) and pick the **SQL Server** connector.
2. Pick the connection `retailhub_sqlserver`.
3. In **Ingestion gateway**, choose the staging catalog and schema for the UC Volume the gateway
   will use (e.g. `<trainer catalog>.lakeflow_staging`). The wizard creates the gateway pipeline.
4. In **Source**, pick database `AdventureWorksLT` and schema `SalesLT`, then
   select the tables `Customer`, `Product`, `SalesOrderHeader` and `SalesOrderDetail`.
5. In **Destination**, pick the trainer's catalog and schema `lakeflow_connect`. Announce both names to the class.
   Participants enter the catalog name in the `trainer_catalog` widget in section D and get `SELECT` on the schema.
6. Set **Schedule** to manual for the demo (production would use a cron trigger), then click **Save and run**.
7. Watch the two pipelines in **Jobs & Pipelines**. The gateway snapshots the 4 tables into the
   staging volume. Then the managed ingestion pipeline materializes them as streaming tables.

8. Let the class read the target, because participants query it in section D. The landed tables are
   ordinary UC tables, so grant the training group `USE CATALOG` on the catalog and
   `USE SCHEMA` + `SELECT` on the target schema.

The first snapshot takes a few minutes. Use this time to recap section A with the class.

## D. Verify the landed tables: run these cells

> **Note:** Run these cells only **after** the trainer's ingestion pipeline has completed its first update.
> The trainer tells you when. Everyone has `SELECT` on the shared target schema.

In real work you check the landed tables right after the first load, before anyone builds on them.
Look at the row counts, a join across tables and the table metadata.

In [0]:
%run ../../setup/00_setup

In [0]:
# Read from the trainer's shared target, not from your personal catalog
dbutils.widgets.text("trainer_catalog", "retailhub_trainer", "Trainer catalog")
LFC_CATALOG = dbutils.widgets.get("trainer_catalog")  # ← the name the trainer announces
LFC_SCHEMA  = "lakeflow_connect"
LFC         = f"{LFC_CATALOG}.{LFC_SCHEMA}"

display(spark.sql(f"SHOW TABLES IN {LFC}"))

In [0]:
# Row counts confirm the first snapshot landed all four SalesLT tables
for t in ["customer", "product", "salesorderheader", "salesorderdetail"]:
    cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {LFC}.{t}").first()["c"]
    print(f"{LFC}.{t:<18} {cnt:>8,} rows")

In [0]:
# Top 10 products by wholesale revenue: the landed tables join like any UC table
display(spark.sql(f"""
    SELECT
        p.Name                                  AS product_name,
        COUNT(DISTINCT h.SalesOrderID)          AS orders,
        SUM(d.OrderQty)                         AS units,
        ROUND(SUM(d.LineTotal), 2)              AS revenue
    FROM {LFC}.salesorderdetail d
    JOIN {LFC}.salesorderheader h ON d.SalesOrderID = h.SalesOrderID
    JOIN {LFC}.product          p ON d.ProductID    = p.ProductID
    GROUP BY p.Name
    ORDER BY revenue DESC
    LIMIT 10
"""))

In [0]:
# Type, Provider and the pipeline properties show a streaming table owned by the ingestion pipeline
display(spark.sql(f"DESCRIBE TABLE EXTENDED {LFC}.customer"))

In [0]:
# Each pipeline update that touched the table is one history entry
display(spark.sql(f"DESCRIBE HISTORY {LFC}.customer LIMIT 5"))

## E. Live CDC round-trip

This round-trip shows that a change in the source reaches bronze through the CDC feed, without a full re-read of the table.

### TRAINER: run this UPDATE in Azure SQL (portal Query editor or `sqlcmd`)

```sql
UPDATE SalesLT.Customer
SET    Phone        = CONCAT('555-', FORMAT(SYSUTCDATETIME(), 'HHmmss')),  -- new value on every run
       ModifiedDate = SYSUTCDATETIME()   -- AdventureWorksLT has no trigger that updates it
WHERE  CustomerID = 1;
```

Then trigger a refresh in **Jobs & Pipelines → ingestion pipeline → Start**. The gateway picks the
change out of the CDC feed, and the ingestion pipeline merges it into the target table.

### Everyone: capture the values before, then re-query after the refresh

Run the first cell before the trainer's UPDATE. Run the second cell after the refresh finishes and compare `Phone` and `ModifiedDate`.

In [0]:
# Run before the trainer's UPDATE: keep the current values to compare after the refresh
before = spark.sql(f"""
    SELECT CustomerID, FirstName, LastName, Phone, ModifiedDate
    FROM {LFC}.customer
    WHERE CustomerID = 1
""").first()
print(f"Before: Phone = {before['Phone']} | ModifiedDate = {before['ModifiedDate']}")

In [0]:
# Run after the refresh: a new Phone or ModifiedDate means the CDC change reached bronze
after = spark.sql(f"SELECT Phone, ModifiedDate FROM {LFC}.customer WHERE CustomerID = 1").first()
print(f"After:  Phone = {after['Phone']} | ModifiedDate = {after['ModifiedDate']}")
if (after["Phone"], after["ModifiedDate"]) != (before["Phone"], before["ModifiedDate"]):
    print("OK: CDC round-trip: source UPDATE → gateway CDC extract → merged into UC bronze")
else:
    print("Change not visible yet — wait for the pipeline refresh to finish and re-run this cell")

## F. Exam call-out and cost note

### Exam (Ingestion, 21%)
- Prioritize by source: files in object storage → Auto Loader; a supported SaaS app or database →
  managed connector; an unsupported source → partner tool; hand-rolled JDBC last.
- A managed connector has three parts: a UC `CONNECTION` (governed credentials) → an ingestion gateway
  (a dedicated pipeline that stages snapshots and CDC changes into a UC Volume) → a managed
  ingestion pipeline (merges the staged changes into streaming tables).
- Database connectors are incremental by design. They consume CDC or change tracking,
  not repeated full extracts. The targets are streaming tables, refreshed by pipeline updates.
- Connections and landed tables are Unity Catalog objects, so normal `GRANT`s, lineage and audit apply.

### Cost note
The gateway pipeline uses classic compute and is designed to run continuously. Stopping it mid-life can force a full re-snapshot.
Delete the ingestion pipeline and the gateway when you no longer need them, instead of leaving them idle.

← <a href="$../../day1/demo/04_incremental_ingestion">04 — Incremental Ingestion (Day 1)</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$./05a_transformations_pyspark">05a — Transformations in PySpark (self-training) →</a>